# Coordinate Check: GQA-muP across r-values

Verifies that activation norms and weight update norms are stable across
different GQA repetition factors (r=1 to r=12) at fixed width 1536,
depth 3, head_dim=128.

**What to look for:**
- `f` (forward): activation/weight norms should be ~constant across r-values
- `Df` (update): weight update norms should be ~constant across r-values
- If norms diverge across r, the muP scaling is wrong for GQA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import re

# Load all CSVs
data_dir = os.path.expanduser("~/Downloads/coord_check_results 2")
files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
print(f"Loading {len(files)} files...")

dfs = []
for f in files:
    try:
        d = pd.read_csv(f, on_bad_lines='skip')
        dfs.append(d)
    except Exception as e:
        print(f"  Skipping {os.path.basename(f)}: {e}")

df = pd.concat(dfs, ignore_index=True)

# Parse r-value from tag
df['r'] = df['tag'].apply(lambda t: int(re.search(r'_r(\d+)_', t).group(1)))
df['kv'] = df['tag'].apply(lambda t: int(re.search(r'_kv(\d+)_', t).group(1)))

print(f"Total rows: {len(df)}")
print(f"r-values: {sorted(df['r'].unique())}")
print(f"Layers: {sorted(df['layer'].unique())}")
print(f"Seeds: {sorted(df['seed'].unique())}")
print(f"Iterations per run: {df['iteration'].max() + 1}")

## Forward pass: activation norms across r

Each subplot is a layer. Lines are different r-values (averaged over seeds).
If GQA-muP is correct, all lines should overlap.

In [ ]:
# Forward activation norms (f, input, norm)
fwd_input = df[(df['fDf'] == 'f') & (df['layer_type'] == 'input') & (df['data_type'] == 'norm')]

layers = sorted(fwd_input['layer'].unique())
r_values = sorted(fwd_input['r'].unique())
colors = plt.cm.viridis([i / (len(r_values) - 1) for i in range(len(r_values))])

fig, axes = plt.subplots(4, 5, figsize=(20, 14), sharex=True)
axes = axes.flatten()

for idx, layer_name in enumerate(layers):
    if idx >= len(axes):
        break
    ax = axes[idx]
    for ci, r in enumerate(r_values):
        sub = fwd_input[(fwd_input['layer'] == layer_name) & (fwd_input['r'] == r)]
        avg = sub.groupby('iteration')['value'].agg(['mean', 'std']).reset_index()
        ax.plot(avg['iteration'], avg['mean'], color=colors[ci], label=f'r={r}', linewidth=1.5)
        ax.fill_between(avg['iteration'], avg['mean'] - avg['std'], avg['mean'] + avg['std'],
                        color=colors[ci], alpha=0.15)
    ax.set_title(layer_name, fontsize=9)
    ax.tick_params(labelsize=8)

# Hide unused axes
for idx in range(len(layers), len(axes)):
    axes[idx].set_visible(False)

axes[0].legend(fontsize=7, loc='upper right')
fig.suptitle('Forward activation norms (should be stable across r)', fontsize=14, y=1.01)
fig.supxlabel('Iteration')
fig.supylabel('Norm')
plt.tight_layout()
plt.show()

## Forward pass: weight norms across r

In [ ]:
# Forward weight norms (f, weight, norm)
fwd_weight = df[(df['fDf'] == 'f') & (df['layer_type'] == 'weight') & (df['data_type'] == 'norm')]

layers_w = sorted(fwd_weight['layer'].unique())

fig, axes = plt.subplots(4, 5, figsize=(20, 14), sharex=True)
axes = axes.flatten()

for idx, layer_name in enumerate(layers_w):
    if idx >= len(axes):
        break
    ax = axes[idx]
    for ci, r in enumerate(r_values):
        sub = fwd_weight[(fwd_weight['layer'] == layer_name) & (fwd_weight['r'] == r)]
        avg = sub.groupby('iteration')['value'].agg(['mean', 'std']).reset_index()
        ax.plot(avg['iteration'], avg['mean'], color=colors[ci], label=f'r={r}', linewidth=1.5)
        ax.fill_between(avg['iteration'], avg['mean'] - avg['std'], avg['mean'] + avg['std'],
                        color=colors[ci], alpha=0.15)
    ax.set_title(layer_name, fontsize=9)
    ax.tick_params(labelsize=8)

for idx in range(len(layers_w), len(axes)):
    axes[idx].set_visible(False)

axes[0].legend(fontsize=7, loc='upper right')
fig.suptitle('Forward weight norms (should be stable across r)', fontsize=14, y=1.01)
fig.supxlabel('Iteration')
fig.supylabel('Norm')
plt.tight_layout()
plt.show()

## Weight updates (Df): do update magnitudes stay constant across r?

This is the key plot. If GQA-muP is working, the weight update norms
should be approximately the same regardless of r. The KV layers
(`attn.c_k`, `attn.c_v`) are the ones most affected by the GQA correction.

In [ ]:
# Weight update norms (Df, weight, norm)
df_weight = df[(df['fDf'] == 'Df') & (df['layer_type'] == 'weight') & (df['data_type'] == 'norm')]

layers_df = sorted(df_weight['layer'].unique())

fig, axes = plt.subplots(4, 5, figsize=(20, 14), sharex=True)
axes = axes.flatten()

for idx, layer_name in enumerate(layers_df):
    if idx >= len(axes):
        break
    ax = axes[idx]
    for ci, r in enumerate(r_values):
        sub = df_weight[(df_weight['layer'] == layer_name) & (df_weight['r'] == r)]
        avg = sub.groupby('iteration')['value'].agg(['mean', 'std']).reset_index()
        ax.plot(avg['iteration'], avg['mean'], color=colors[ci], label=f'r={r}', linewidth=1.5)
        ax.fill_between(avg['iteration'], avg['mean'] - avg['std'], avg['mean'] + avg['std'],
                        color=colors[ci], alpha=0.15)
    ax.set_title(layer_name, fontsize=9)
    ax.tick_params(labelsize=8)

for idx in range(len(layers_df), len(axes)):
    axes[idx].set_visible(False)

axes[0].legend(fontsize=7, loc='upper right')
fig.suptitle('Weight update norms (Df) — should be stable across r', fontsize=14, y=1.01)
fig.supxlabel('Iteration')
fig.supylabel('Update norm')
plt.tight_layout()
plt.show()

## Summary: KV layers specifically

Zoom in on `attn.c_k` and `attn.c_v` — these are the layers where
the GQA correction factor `(r + r^½) / 2m` matters. Compare forward
norms, weight norms, and update norms side by side.

In [ ]:
# Focus on KV layers: c_k and c_v across all 3 transformer layers
kv_layers = [l for l in df['layer'].unique() if 'c_k' in l or 'c_v' in l]
kv_layers = sorted(kv_layers)

# Plot: rows = (f input, f weight, Df weight), cols = KV layers
combos = [
    ('f', 'input', 'norm', 'Activation norm'),
    ('f', 'weight', 'norm', 'Weight norm'),
    ('Df', 'weight', 'norm', 'Weight update norm'),
]

fig, axes = plt.subplots(len(combos), len(kv_layers), figsize=(3.5 * len(kv_layers), 3.5 * len(combos)),
                          sharex=True)

for row, (fdf, lt, dt, row_label) in enumerate(combos):
    sub = df[(df['fDf'] == fdf) & (df['layer_type'] == lt) & (df['data_type'] == dt)]
    for col, layer_name in enumerate(kv_layers):
        ax = axes[row, col]
        layer_sub = sub[sub['layer'] == layer_name]
        for ci, r in enumerate(r_values):
            rsub = layer_sub[layer_sub['r'] == r]
            avg = rsub.groupby('iteration')['value'].agg(['mean', 'std']).reset_index()
            ax.plot(avg['iteration'], avg['mean'], color=colors[ci], label=f'r={r}', linewidth=2)
            ax.fill_between(avg['iteration'], avg['mean'] - avg['std'], avg['mean'] + avg['std'],
                            color=colors[ci], alpha=0.15)
        if row == 0:
            ax.set_title(layer_name, fontsize=11)
        if col == 0:
            ax.set_ylabel(row_label, fontsize=11)
        ax.tick_params(labelsize=9)

axes[0, -1].legend(fontsize=8, loc='upper right')
fig.suptitle('KV layer coordinate check: GQA-muP across r-values', fontsize=14, y=1.01)
fig.supxlabel('Iteration')
plt.tight_layout()
plt.show()

## Final iteration snapshot: norm vs r

Bar chart showing the norm at the final iteration for each layer,
grouped by r. This is the clearest view of whether scaling is stable.

In [ ]:
# Snapshot at final iteration: weight update norms across r for all layers
# KV layers normalized by (1+√r), non-KV layers shown raw

df_updates = df[(df['fDf'] == 'Df') & (df['layer_type'] == 'weight') & (df['data_type'] == 'norm')]
last_df_iter = df_updates['iteration'].max()
snapshot = df_updates[df_updates['iteration'] == last_df_iter].copy()

# Normalize KV layers by (1+√r)
kv_mask = snapshot['layer'].str.contains('c_k|c_v')
snapshot.loc[kv_mask, 'value'] = snapshot.loc[kv_mask, 'value'] / (1 + np.sqrt(snapshot.loc[kv_mask, 'r']))

avg_snapshot = snapshot.groupby(['layer', 'r'])['value'].agg(['mean', 'std']).reset_index()
layers_plot = sorted(avg_snapshot['layer'].unique())

fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(layers_plot))
width = 0.12
offsets = np.linspace(-width * (len(r_values)-1)/2, width * (len(r_values)-1)/2, len(r_values))

for ci, r in enumerate(r_values):
    rsub = avg_snapshot[avg_snapshot['r'] == r]
    vals = []
    errs = []
    for l in layers_plot:
        row = rsub[rsub['layer'] == l]
        vals.append(row['mean'].values[0] if len(row) > 0 else 0)
        errs.append(row['std'].values[0] if len(row) > 0 else 0)
    ax.bar(x + offsets[ci], vals, width, yerr=errs, label=f'r={r}',
           color=colors[ci], alpha=0.85, capsize=2)

ax.set_xticks(x)
ax.set_xticklabels(layers_plot, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Weight update norm (final iter)')
ax.set_title('Weight update norms at final iteration — KV layers normalized by (1+√r)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Coordinate check: r on x-axis, iterations in color

Each subplot is a layer. X-axis is r (GQA ratio). Lines are iterations.
If GQA-muP is correct, lines should be flat across r.

In [ ]:
import numpy as np

# --- Helper: plot with r on x-axis, iterations as colored lines ---
def plot_r_vs_norm(data_subset, title, layers_to_plot=None):
    if layers_to_plot is None:
        layers_to_plot = sorted(data_subset['layer'].unique())
    
    iters = sorted(data_subset['iteration'].unique())
    n_iters = len(iters)
    iter_colors = plt.cm.plasma(np.linspace(0, 0.9, n_iters))
    
    ncols = min(5, len(layers_to_plot))
    nrows = (len(layers_to_plot) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)
    axes_flat = axes.flatten()
    
    for idx, layer_name in enumerate(layers_to_plot):
        ax = axes_flat[idx]
        layer_data = data_subset[data_subset['layer'] == layer_name]
        
        for ci, it in enumerate(iters):
            it_data = layer_data[layer_data['iteration'] == it]
            avg = it_data.groupby('r')['value'].agg(['mean', 'std']).reset_index()
            avg = avg.sort_values('r')
            label = f't={it}' if ci % max(1, n_iters // 5) == 0 else None
            ax.plot(avg['r'], avg['mean'], color=iter_colors[ci], linewidth=1.5,
                    label=label, marker='o', markersize=3)
            ax.fill_between(avg['r'], avg['mean'] - avg['std'], avg['mean'] + avg['std'],
                            color=iter_colors[ci], alpha=0.1)
        
        ax.set_title(layer_name, fontsize=10)
        ax.set_xlabel('r')
        ax.set_xscale('log', base=2)
        ax.set_xticks(sorted(data_subset['r'].unique()))
        ax.set_xticklabels([str(r) for r in sorted(data_subset['r'].unique())])
    
    for idx in range(len(layers_to_plot), len(axes_flat)):
        axes_flat[idx].set_visible(False)
    
    axes_flat[0].legend(fontsize=7, loc='best')
    fig.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

# --- Forward activation norms ---
fwd_input = df[(df['fDf'] == 'f') & (df['layer_type'] == 'input') & (df['data_type'] == 'norm')]
plot_r_vs_norm(fwd_input, 'Forward activation norms vs r (should be flat)')

# --- Forward weight norms ---
fwd_weight = df[(df['fDf'] == 'f') & (df['layer_type'] == 'weight') & (df['data_type'] == 'norm')]
plot_r_vs_norm(fwd_weight, 'Forward weight (spectral) norms vs r (should be flat)')

# --- Weight update norms ---
df_update = df[(df['fDf'] == 'Df') & (df['layer_type'] == 'weight') & (df['data_type'] == 'norm')]
plot_r_vs_norm(df_update, 'Weight update norms vs r (should be flat)')

## KV layers only: r on x-axis

In [ ]:
# KV layers only
kv_layers = sorted([l for l in df['layer'].unique() if 'c_k' in l or 'c_v' in l])

fwd_input_kv = fwd_input[fwd_input['layer'].isin(kv_layers)]
plot_r_vs_norm(fwd_input_kv, 'KV activation norms vs r (raw)', kv_layers)

fwd_weight_kv = fwd_weight[fwd_weight['layer'].isin(kv_layers)]
plot_r_vs_norm(fwd_weight_kv, 'KV weight norms vs r (raw)', kv_layers)

df_update_kv = df_update[df_update['layer'].isin(kv_layers)]
plot_r_vs_norm(df_update_kv, 'KV weight update norms vs r (raw)', kv_layers)

# --- Normalized by 1/(1+√r) ---
# measured = σ_max(W) × √r (natural norm includes aspect ratio)
# σ_max(W) = Θ((1+√r)/√r)  =>  measured = Θ(1+√r)
# So dividing by (1+√r) should flatten

fwd_input_kv_norm = fwd_input_kv.copy()
fwd_input_kv_norm['value'] = fwd_input_kv_norm['value'] / (1 + np.sqrt(fwd_input_kv_norm['r']))
plot_r_vs_norm(fwd_input_kv_norm, 'KV activation norms / (1+√r)', kv_layers)

fwd_weight_kv_norm = fwd_weight_kv.copy()
fwd_weight_kv_norm['value'] = fwd_weight_kv_norm['value'] / (1 + np.sqrt(fwd_weight_kv_norm['r']))
plot_r_vs_norm(fwd_weight_kv_norm, 'KV weight norms / (1+√r)', kv_layers)

df_update_kv_norm = df_update_kv.copy()
df_update_kv_norm['value'] = df_update_kv_norm['value'] / (1 + np.sqrt(df_update_kv_norm['r']))
plot_r_vs_norm(df_update_kv_norm, 'KV weight update norms / (1+√r)', kv_layers)

## Debugging: attn.c_v.2 weight update norms vs r

Pull just this layer's Df weight norms and experiment with different
normalization factors to figure out the correct r-dependence.

In [ ]:
# Extract attn.c_v.2 weight update norms
cv2 = df[(df['layer'] == 'attn.c_v.2') & (df['fDf'] == 'Df') & 
         (df['layer_type'] == 'weight') & (df['data_type'] == 'norm')]

# Average over seeds, take last iteration
last_it = cv2['iteration'].max()
cv2_last = cv2[cv2['iteration'] == last_it].groupby('r')['value'].agg(['mean', 'std']).reset_index()
cv2_last.columns = ['r', 'mean', 'std']

print("attn.c_v.2 weight update norm (Df) at final iteration:")
print(cv2_last.to_string(index=False))
print()

# Current muP kv lr_scale = (1 + r^½) / (2m), m=6
m = 6.0
cv2_last['lr_scale'] = (1 + np.sqrt(cv2_last['r'])) / (2 * m)
cv2_last['raw_over_lr'] = cv2_last['mean'] / cv2_last['lr_scale']
print("Dividing by lr_scale = (1+√r)/(2m):")
print(cv2_last[['r', 'mean', 'lr_scale', 'raw_over_lr']].to_string(index=False))

In [ ]:
# Playground: try different normalization factors
#
# What we measure:  natural_norm = σ_max(W) × √r   (aspect ratio correction)
# What we expect:   σ_max(W) = Θ((1+√r)/√r)
# Therefore:        natural_norm = (1+√r)/√r × √r = (1+√r)
#
# So dividing measured by (1+√r) should flatten it.

r_vals = cv2_last['r'].values
raw = cv2_last['mean'].values

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

normalizations = {
    'raw  [= σ_max × √r]':                 lambda r: 1,
    '/ √r  [= σ_max]':                     lambda r: np.sqrt(r),
    '/ (1+√r)  [predicted flat if\n'
    '  σ_max = Θ((1+√r)/√r)]':             lambda r: (1+np.sqrt(r)),
    '× √r/(1+√r)  [predicted flat if\n'
    '  σ_max = Θ((1+√r)/√r)]':             lambda r: (1+np.sqrt(r))/np.sqrt(r),
    '/ √r(1+√r)/√r  [= σ_max/prediction\n'
    '  should be Θ(1)]':                    lambda r: np.sqrt(r) * (1+np.sqrt(r))/np.sqrt(r),
    'CUSTOM: edit me':                       lambda r: 1,  # <-- EDIT THIS
}

for ax, (name, norm_fn) in zip(axes.flatten(), normalizations.items()):
    normed = raw / np.array([norm_fn(r) for r in r_vals])
    ax.plot(r_vals, normed, 'o-', linewidth=2, markersize=8)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('r')
    ax.set_ylabel('normalized')
    ax.set_xscale('log', base=2)
    ax.set_xticks(r_vals)
    ax.set_xticklabels([str(r) for r in r_vals])
    cv = np.std(normed) / np.mean(normed) * 100
    ax.text(0.95, 0.95, f'CV={cv:.1f}%', transform=ax.transAxes, 
            ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('attn.c_v.2 Df norm: measured = σ_max(ΔW) × √r', fontsize=13)
plt.tight_layout()
plt.show()

# Decomposition table
print("Decomposition: measured = σ_max × √r,  predicted σ_max = C×(1+√r)/√r")
print()
print(f"{'r':>4} | {'measured':>10} | {'σ_max':>10} | {'(1+√r)/√r':>10} | {'σ_max/pred':>10} | {'meas/(1+√r)':>12}")
print("-" * 72)
for r, v in zip(r_vals, raw):
    sigma = v / np.sqrt(r)
    pred = (1 + np.sqrt(r)) / np.sqrt(r)
    ratio = sigma / pred
    meas_over = v / (1 + np.sqrt(r))
    print(f"{r:>4} | {v:>10.6f} | {sigma:>10.6f} | {pred:>10.4f} | {ratio:>10.6f} | {meas_over:>12.6f}")